# Создание Эмбеддинга

**Эмбеддинг** - представление дискретных объектов в виде плотных векторов фиксированной размерности.

In [7]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np, random

In [2]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [3]:
# Параметры
vocab_size = 10000  # Количество уникальных объектов (например, слов в словаре)
embedding_dim = 100  # Размерность векторов

In [4]:
# слой Эмбеддинга
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

In [5]:
# Пример индекса. Пусть 42 кодирует слово "кошка"
index = torch.tensor([42])

embedded_vector = embedding_layer(index)
print("Размерность вектора:", embedded_vector.shape)
print("Вектор для индекса 42:", embedded_vector)

Размерность вектора: torch.Size([1, 100])
Вектор для индекса 42: tensor([[ 1.9905, -1.1816,  0.3039,  2.2259,  0.3058, -0.6298,  0.7762, -0.5106,
          0.3001,  0.1731,  0.0859,  0.1077,  1.0630,  1.8410, -0.5687, -1.4458,
         -0.2562,  1.1728, -0.4283, -0.2986,  0.1489,  1.6612, -2.6223, -0.4319,
         -0.1010, -0.4388, -1.9775, -0.0359, -0.7873, -0.1165,  1.8994,  0.4524,
         -0.5850,  0.1741, -0.2031, -1.2714,  0.8729,  0.1414, -1.4058,  0.4083,
          1.4583, -0.5316,  0.8971, -1.1392, -1.3100, -0.0962, -2.4108, -0.0096,
         -0.6613,  0.8029, -0.0705, -1.5173,  0.2003, -0.2469,  0.2363,  0.7618,
         -0.5063,  0.7493,  1.0682, -2.6366,  0.6954, -0.3230,  0.5970, -0.6248,
          1.6473, -1.7078,  0.0963, -0.2865,  1.0168, -0.6580, -0.3999, -1.7974,
         -0.3784,  0.2324, -0.7322, -0.4037,  1.2063, -0.6938, -1.5165,  2.0242,
         -0.8075,  0.0941,  0.2730,  0.5441, -2.4758,  0.2106,  0.2536, -0.7318,
         -1.3799, -0.2103,  0.8919,  0.1832,

Применение на батче

In [ ]:
# Допустим, у нас два предложения, каждое из четырёх токенов (индексы слов)
batch_indices = torch.tensor([
    [12, 42, 7, 89],
    [5, 999, 123, 42]
])

print("Размер входного батча:", batch_indices.shape)

# Применяем Embedding
embedded_batch = embedding_layer(batch_indices)

print("Выходной размер:", embedded_batch.shape)

print("Эмбеддинг слова с индексом 42:")
print(embedding_layer(torch.tensor([42])))


# (batch_size, seq_len) -> (batch_size, seq_len, embedding_dim)
print("Итоговая размерность:", embedded_batch.shape)

Размер входного батча: torch.Size([2, 4])
Выходной размер: torch.Size([2, 4, 100])
Эмбеддинг слова с индексом 42:
tensor([[ 1.9905, -1.1816,  0.3039,  2.2259,  0.3058, -0.6298,  0.7762, -0.5106,
          0.3001,  0.1731,  0.0859,  0.1077,  1.0630,  1.8410, -0.5687, -1.4458,
         -0.2562,  1.1728, -0.4283, -0.2986,  0.1489,  1.6612, -2.6223, -0.4319,
         -0.1010, -0.4388, -1.9775, -0.0359, -0.7873, -0.1165,  1.8994,  0.4524,
         -0.5850,  0.1741, -0.2031, -1.2714,  0.8729,  0.1414, -1.4058,  0.4083,
          1.4583, -0.5316,  0.8971, -1.1392, -1.3100, -0.0962, -2.4108, -0.0096,
         -0.6613,  0.8029, -0.0705, -1.5173,  0.2003, -0.2469,  0.2363,  0.7618,
         -0.5063,  0.7493,  1.0682, -2.6366,  0.6954, -0.3230,  0.5970, -0.6248,
          1.6473, -1.7078,  0.0963, -0.2865,  1.0168, -0.6580, -0.3999, -1.7974,
         -0.3784,  0.2324, -0.7322, -0.4037,  1.2063, -0.6938, -1.5165,  2.0242,
         -0.8075,  0.0941,  0.2730,  0.5441, -2.4758,  0.2106,  0.2536, -0.7

### Применение к табличным данным

In [ ]:
# Создаем датафрейм

df = pd.DataFrame({
    "city": [
        "Москва",
        "СПб",
        "Казань",
        "Москва",
        "Новосибирск"
    ],
    "weight": [2.1, 5.3, 1.8, 7.0, 4.2],
    "price": [350, 520, 280, 610, 430]
})

In [ ]:
# преобразования
# ---
df["city_id"] = pd.Categorical(df["city"]).codes

# Получаем тензон городов
city_tensor = torch.tensor(df["city_id"].values, dtype=torch.long)

# Получаем тензон весов
weight_tensor = torch.tensor(
    df["weight"].values,
    dtype=torch.float32
).unsqueeze(1)

# Получаем тензон цен
target = torch.tensor(
    df["price"].values,
    dtype=torch.float32
).unsqueeze(1)

In [ ]:
# Создаем эмбеддинги
embedding = nn.Embedding(
    num_embeddings=df["city_id"].nunique(),  # число уникальных городов
    embedding_dim=4                          # размерность эмбеддинга
)

city_embeddings = embedding(city_tensor)

print(city_embeddings.shape)

torch.Size([5, 4])


### Применение выравнивания

In [ ]:
from torch.nn.utils.rnn import pad_sequence

# Пример батча с текстами разной длины
batch = [
    torch.tensor([0, 1, 2]),
    torch.tensor([3, 4]),
    torch.tensor([5, 6, 7, 8])
]

# Применение padding'а
padded_batch = pad_sequence(batch, batch_first=True)
print(padded_batch)

tensor([[0, 1, 2, 0],
        [3, 4, 0, 0],
        [5, 6, 7, 8]])


### Усреднение с помощью Mean Polling

In [ ]:
# Допустим, у нас есть эмбеддинги для трёх слов 
# (batch_size=1, seq_len=3, embedding_dim=5)
embeddings = torch.tensor([[
    [0.1, 0.2, 0.3, 0.4, 0.5],  # Эмбеддинг первого слова
    [0.6, 0.7, 0.8, 0.9, 1.0],  # Эмбеддинг второго слова
    [1.1, 1.2, 1.3, 1.4, 1.5]   # Эмбеддинг третьего слова
]])

# Применяем mean pooling по последовательности (dim=1)
pooled = torch.mean(embeddings, dim=1)

print(pooled.shape)  # torch.Size([1, 5]) — 
# усреднённый вектор размерности 5
print(pooled)

torch.Size([1, 5])
tensor([[0.6000, 0.7000, 0.8000, 0.9000, 1.0000]])


### Сборка модели

In [ ]:
class TextMLP(nn.Module): #  Объявление класса модели
    def __init__(
        self,
        vocab_size: int,  # размер словаря
        embedding_dim: int,  # размерность эмбеддингов
        hidden_dim: int,  # размерность скрытых слоёв MLP
        num_classes: int,  # количество классов
        padding_idx: int = 0,  # индекс паддинг-токена
        dropout: float = 0.2,  # вероятность дропаута
    ):
        super().__init__()
        
        # Объявление эмбеддинга
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=padding_idx,
        )
        
        # Объявление слоёв перцептрона
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, input_ids):
        """
        input_ids: [batch_size, seq_len]
        """

        # [batch_size, seq_len, embedding_dim]
        embeddings = self.embedding(input_ids)
        # Объявление pooling'а
        pooled = embeddings.mean(dim=1)

        # [batch_size, num_classes]
        logits = self.mlp(pooled)

        return logits



In [ ]:
# Объявление модели
model = TextMLP(
    vocab_size=50000,
    embedding_dim=128,
    hidden_dim=256,
    num_classes=5,
)

# Создание батча данных (пример)
batch = torch.randint(
    low=0,
    high=50000,
    size=(32, 100)
)

logits = model(batch) # Получение выходного вектора

print(f"Размер выходного вектора: {logits.shape}")

Размер выходного вектора: torch.Size([32, 5])


In [15]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader


def collate_fn(batch, pad_token_id=0): 
    """Объединение и предобработка батчей"""
    input_ids = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    
    # Дополняем последовательности до максимальной длины в батче
    padded_sequences = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=pad_token_id
    )

    return {
        "input_ids": padded_sequences,
        "labels": labels
    }


# Dataset
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "input_ids": self.texts[idx],
            "label": self.labels[idx]
        }


In [16]:
#  Объявляем тензоры текстов
texts = [
    [5, 12, 31, 18],
    [7, 9],
    [11, 25, 77, 13, 42]
]

#  Объявляем лейблы для классификации
labels = [0, 1, 0]

# Загружаем данные
dataset = TextDataset(texts, labels)

train_loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

for i, batch in enumerate(train_loader):
    print(f"\n=== БАТЧ №{i} В ЦИКЛЕ ОБУЧЕНИЯ ===")
    print(f"Тензор input_ids:\n{batch['input_ids']}")
    print(f"Тензор labels: {batch['labels']}")
    print(f"Тип данных: {batch['input_ids'].dtype}")


=== БАТЧ №0 В ЦИКЛЕ ОБУЧЕНИЯ ===
Тензор input_ids:
tensor([[11, 25, 77, 13, 42],
        [ 5, 12, 31, 18,  0]])
Тензор labels: tensor([0, 0])
Тип данных: torch.int64

=== БАТЧ №1 В ЦИКЛЕ ОБУЧЕНИЯ ===
Тензор input_ids:
tensor([[7, 9]])
Тензор labels: tensor([1])
Тип данных: torch.int64


In [17]:
# Объявляем пример тензора с векторами текстов
texts = [
    [5, 12, 31, 18],
    [7, 9],
    [11, 25, 77, 13, 42]
] 

# Объявляем лейблы для каждого текста
labels = [0, 1, 0]

# Формируем и загружаем датасет
dataset = TextDataset(texts, labels)

train_loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

# Создаём экземпляр модели с параметрами
model = TextMLP(
    vocab_size=100000,
    embedding_dim=128,
    hidden_dim=256,
    num_classes=2
)

# Инициализируем оптимизатор и criterion
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Проводим обучение на всех текстах
for batch in train_loader:
    optimizer.zero_grad()
    logits = model(batch["input_ids"])

    loss = criterion(logits, batch["labels"])
    loss.backward()
    optimizer.step()

    print(loss.item())

0.6733717918395996
0.6434327960014343


### Ограничения модели на основе Embedding + Mean Pooling + MLP 

- Потеря порядка слов. Mean Pooling усредняет векторы всех токенов, полностью игнорируя порядок слов в тексте.
- Потеря контекста и семантических связей. Mean Pooling не учитывает взаимосвязь между словами. 
- Чувствительность к длине текста. Mean Pooling усредняет все векторы токенов, что может размывать важную информацию в длинных текстах. Чем длинее текст, тем больше размытие